In [1]:
import os
import sys
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch

project_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
sys.path.append(project_dir)
from raw_data_processing import get_x, get_y, get_wavelength
from tools import JSON_Read#, plotly_multi_scatter
from tools import KAN_es_2 as KAN_es
from tools import lmdKAN_es
SCRIPT_DIR = os.path.abspath('')


def get_sqz_input(x_axis, y_axis):
    ''' Evaluate squeezed data from curve.
    '''
    i_max = np.argmax(y_axis)
    I = y_axis[i_max]  # Max I
    c_I = x_axis[i_max]  # Coordinate of max I

    diff_I = np.absolute(y_axis-I/2)
    c_I2_left = x_axis[ np.argmin(diff_I[:i_max]) ]  # Left I/2 coordinate
    c_I2_right = x_axis[ np.argmin(diff_I[i_max:])+i_max]  # Right I/2 coordinate

    c_I2 = np.mean([c_I2_left, c_I2_right])  # Mean center coordinate on I/2 
    disp_I2 = np.abs(c_I2_right-c_I2_left)  # Width of curve on I/2 y-level

    integr_ratio = np.sum(y_axis[i_max+1:]) / np.sum(y_axis[:i_max])

    sqz_input = [I, c_I, c_I2, disp_I2, integr_ratio]
    
    
    return sqz_input

def get_all_sqz_input(matr_x, matr_y):
    matr_sqz_input = []
    for x_axis, y_axis in zip(matr_x, matr_y):
        matr_sqz_input.append(get_sqz_input(x_axis, y_axis))


    return np.array(matr_sqz_input)


d_config = JSON_Read("", "json_config.txt")
EXCITE_WAVE_LENGTH = d_config['EXCITE_WAVE_LENGTH']
#PREDICT_IONS = ['Cr']# ['Cr'], ['Cu'], ['Ni'], ['NO3']# d_config['PREDICT_IONS']
SPEC_FOLDER = d_config['SPEC_FOLDER']

TRAIN_TEST_RATIO = d_config['TRAIN_TEST_RATIO']
VALIDATION_TRAIN_RATIO = d_config['VALIDATION_TRAIN_RATIO']
N_ITER_NO_CHANGE = d_config['N_ITER_NO_CHANGE']

HIDDEN_LAYER_SIZES = d_config['HIDDEN_LAYER_SIZES']
ACTIVATION = d_config['ACTIVATION']
SOLVER = d_config['SOLVER']
MAX_ITER = d_config['MAX_ITER']
TOL = d_config['TOL']

x = get_x(wave_length=EXCITE_WAVE_LENGTH, spec_file=project_dir+'\\'+SPEC_FOLDER)

y_Cr = get_y(l_ions='Cr', spec_file=project_dir+'\\'+SPEC_FOLDER)
y_Cu = get_y(l_ions='Cu', spec_file=project_dir+'\\'+SPEC_FOLDER)
y_Ni = get_y(l_ions='Ni', spec_file=project_dir+'\\'+SPEC_FOLDER)
y_NO3 = get_y(l_ions='NO3', spec_file=project_dir+'\\'+SPEC_FOLDER)

l_wavelenth = get_wavelength(spec_file=project_dir+'\\'+SPEC_FOLDER)

x_matrix, y_matrix = np.broadcast_to(l_wavelenth, (len(x), len(l_wavelenth))), x.to_numpy()
x = get_all_sqz_input(x_matrix, y_matrix)

In [2]:
from sklearn.base import RegressorMixin, BaseEstimator, _fit_context
from sklearn.utils.validation import check_is_fitted


class KANRegressor(RegressorMixin, BaseEstimator):
    """Sci-kit learn wrapper for pykan model.
    
    Hierarchical inheritance chain of classes:
    1. pykan.kan --> 
    2. --> KAN_es: pykan.kan with early stopping, made as in skl.neural_network.MLPRegressor -->
    3. --> KANRegressor: pykan.kan wrapped in (RegressorMixin, BaseEstimator) for compatibility with skl interface.
           Uses params, inspired by skl.neural_network.MLPRegressor params logic.
    """

    # This is a dictionary allowing to define the type of parameters.
    # It used to validate parameter within the `_fit_context` decorator.
    
    _parameter_constraints = {}
    
    _d_solver_translation = {
        'lbfgs': 'LBFGS',
        'adam': 'Adam'
    }

    def __init__(self, 
                 # kan.__init__ args
                 hidden_layer_sizes=None,
                 grid=3,
                 k=3,
                 seed=1,
                 device='cpu',
                 # kan.train_es args
                 tol=1e-3,
                 n_iter_no_change=10,
                 solver='lbfgs',
                 max_iter=100,
                 learning_rate_init=1.0,
                 validation_fraction=0.1,
                 # additional kan.__init__ args
                 mult_arity = 2, noise_scale=0.3, scale_base_mu=0.0, scale_base_sigma=1.0, base_fun='silu', symbolic_enabled=True, affine_trainable=False, grid_eps=0.02, grid_range=[-1, 1], sp_trainable=True, sb_trainable=True, save_act=True, sparse_init=False, auto_save=True, first_init=True, ckpt_path='./model', state_id=0, round=0,
                 # additional kan.train_es args
                 log=1, lamb=0., lamb_l1=1., lamb_entropy=2., lamb_coef=0., lamb_coefdiff=0., update_grid=True, grid_update_num=10, loss_fn=None, start_grid_update_step=-1, stop_grid_update_step=50, batch=-1, metrics=None, singularity_avoiding=False, save_fig=False, in_vars=None, out_vars=None, beta=3, save_fig_freq=1, img_folder='./video',  y_th=1000., reg_metric='edge_forward_spline_n', display_metrics=None
                 ):
        '''
        Init method. Saving all kan.__init__ and kan.fit args.
        
        Args:
        -----
            -- kan.__init__ parameteres --
            hidden_layer_sizes : list
                the ith element represents the number of neurons in the ith hidden layer.
            grid : int
                number of grid intervals. Default: 3.
            k : int
                order of piecewise polynomial. Default: 3.
            mult_arity : int, or list of int lists
                multiplication arity for each multiplication node (the number of numbers to be multiplied)
            noise_scale : float
                initial injected noise to spline.
            base_fun : str
                the residual function b(x). Default: 'silu'
            symbolic_enabled : bool
                compute (True) or skip (False) symbolic computations (for efficiency). By default: True. 
            affine_trainable : bool
                affine parameters are updated or not. Affine parameters include node_scale, node_bias, subnode_scale, subnode_bias
            grid_eps : float
                When grid_eps = 1, the grid is uniform; when grid_eps = 0, the grid is partitioned using percentiles of samples. 0 < grid_eps < 1 interpolates between the two extremes.
            grid_range : list/np.array of shape (2,))
                setting the range of grids. Default: [-1,1]. This argument is not important if fit(update_grid=True) (by default updata_grid=True)
            sp_trainable : bool
                If true, scale_sp is trainable. Default: True.
            sb_trainable : bool
                If true, scale_base is trainable. Default: True.
            device : str
                device
            seed : int
                random seed
            save_act : bool
                indicate whether intermediate activations are saved in forward pass
            sparse_init : bool
                sparse initialization (True) or normal dense initialization. Default: False.
            auto_save : bool
                indicate whether to automatically save a checkpoint once the model is modified
            state_id : int
                the state of the model (used to save checkpoint)
            ckpt_path : str
                the folder to store checkpoints. Default: './model'
            round : int
                the number of times rewind() has been called
            
            -- kan.train_es parameteres --
            tol : float
                Delta of validation fit which doesn`t count as fitness improvement. (Tolerence of training).
            n_iter_no_change : int
                Number of iteration with no fit change to early stopping.
            solver : str
                "lbfgs" or "adam"
            max_iter : int
                training steps
            learning_rate_init: float
                learning rate
            log : int
                logging frequency
            lamb : float
                overall penalty strength
            lamb_l1 : float
                l1 penalty strength
            lamb_entropy : float
                entropy penalty strength
            lamb_coef : float
                coefficient magnitude penalty strength
            lamb_coefdiff : float
                difference of nearby coefficits (smoothness) penalty strength
            update_grid : bool
                If True, update grid regularly before stop_grid_update_step
            grid_update_num : int
                the number of grid updates before stop_grid_update_step
            stop_grid_update_step : int
                no grid updates after this training step
            batch : int
                batch size, if -1 then full.
            small_mag_threshold : float
                threshold to determine large or small numbers (may want to apply larger penalty to smaller numbers)
            small_reg_factor : float
                penalty strength applied to small factors relative to large factos
            save_fig_freq : int
                save figure every (save_fig_freq) step
        '''
        
        self.hidden_layer_sizes, self.grid, self.k, self.seed, self.device = hidden_layer_sizes, grid, k, seed, device
        self.mult_arity, self.noise_scale, self.scale_base_mu, self.scale_base_sigma, self.base_fun, self.symbolic_enabled, self.affine_trainable, self.grid_eps, self.grid_range, self.sp_trainable, self.sb_trainable, self.save_act, self.sparse_init, self.auto_save, self.first_init, self.ckpt_path, self.state_id, self.round = mult_arity, noise_scale, scale_base_mu, scale_base_sigma, base_fun, symbolic_enabled, affine_trainable, grid_eps, grid_range, sp_trainable, sb_trainable, save_act, sparse_init, auto_save, first_init, ckpt_path, state_id, round
        
        self.tol, self.n_iter_no_change, self.solver, self.max_iter, self.learning_rate_init, self.validation_fraction = tol, n_iter_no_change, solver, max_iter, learning_rate_init, validation_fraction
        self.log, self.lamb, self.lamb_l1, self.lamb_entropy, self.lamb_coef, self.lamb_coefdiff, self.update_grid, self.grid_update_num, self.loss_fn, self.start_grid_update_step, self.stop_grid_update_step, self.batch, self.metrics, self.singularity_avoiding, self.save_fig, self.in_vars, self.out_vars, self.beta, self.save_fig_freq, self.img_folder, self.y_th, self.reg_metric, self.display_metrics = log, lamb, lamb_l1, lamb_entropy, lamb_coef, lamb_coefdiff, update_grid, grid_update_num, loss_fn, start_grid_update_step, stop_grid_update_step, batch, metrics, singularity_avoiding, save_fig, in_vars, out_vars, beta, save_fig_freq, img_folder, y_th, reg_metric, display_metrics
        
    @_fit_context(prefer_skip_nested_validation=True)
    def fit(self, X, y, speed=False):
        """A reference implementation of a fitting function.

        Parameters
        ----------
        X : {array-like, sparse matrix}, shape (n_samples, n_features)
            The training input samples.

        y : array-like, shape (n_samples,) or (n_samples, n_outputs)
            The target values (class labels in classification, real numbers in
            regression).

        Returns
        -------
        self : object
            Returns self.
        """
        # `_validate_data` is defined in the `BaseEstimator` class.
        # It allows to:
        # - run different checks on the input data;
        # - define some attributes associated to the input data: `n_features_in_` and
        #   `feature_names_in_`.
        
        # Make sure self.hidden_layer_sizes is a list
        hidden_layer_sizes = self.hidden_layer_sizes
        if not hasattr(hidden_layer_sizes, "__iter__"):
            hidden_layer_sizes = [hidden_layer_sizes]
        hidden_layer_sizes = list(hidden_layer_sizes)
        
        X, y = self._validate_data(X, y, accept_sparse=True)
        n_features = X.shape[1]
        
        # Ensure y is 2D
        if y.ndim == 1:
            y = y.reshape((-1, 1))
            
        if not hasattr(self, "is_fitted_"): self.is_fitted_ = False
        
        if not self.is_fitted_:
            self.n_outputs_ = y.shape[1]
        
            self.width = [n_features] + self.hidden_layer_sizes + [self.n_outputs_]
            self.kan = KAN_es(width=self.width, grid=self.grid, k=self.k, seed=self.seed, device=self.device,
                              mult_arity=self.mult_arity, noise_scale=self.noise_scale, scale_base_mu=self.scale_base_mu, scale_base_sigma=self.scale_base_sigma, base_fun=self.base_fun, symbolic_enabled=self.symbolic_enabled, affine_trainable=self.affine_trainable, grid_eps=self.grid_eps, grid_range=self.grid_range, sp_trainable=self.sp_trainable, sb_trainable=self.sb_trainable, save_act=self.save_act, sparse_init=self.sparse_init, auto_save=self.auto_save, first_init=self.first_init, ckpt_path=self.ckpt_path, state_id=self.state_id, round=self.round)
        
        if speed: self.kan.speed()
        
        X_train, X_val, y_train, y_val = train_test_split(X, y, 
                                                        test_size=self.validation_fraction,
                                                        random_state=self.seed)
                    
        kan_dataset = {'train_input': torch.tensor(np.array(X_train), dtype=torch.float),
                       'train_label': torch.tensor(np.array(y_train), dtype=torch.float),
                       'test_input': torch.tensor(np.array(X_val), dtype=torch.float),
                       'test_label': torch.tensor(np.array(y_val), dtype=torch.float)}
        
        results = self.kan.fit(kan_dataset, 
                               tol=self.tol, 
                               n_iter_no_change=self.n_iter_no_change,
                               opt=self._d_solver_translation[self.solver], 
                               lr = self.learning_rate_init,
                               steps=self.max_iter,
                               log=self.log, lamb=self.lamb, lamb_l1=self.lamb_l1, lamb_entropy=self.lamb_entropy, lamb_coef=self.lamb_coef, lamb_coefdiff=self.lamb_coefdiff, update_grid=self.update_grid, grid_update_num=self.grid_update_num, loss_fn=self.loss_fn, stop_grid_update_step=self.stop_grid_update_step, batch=self.batch, singularity_avoiding=self.singularity_avoiding, save_fig=self.save_fig, in_vars=self.in_vars, out_vars=self.out_vars, beta=self.beta, save_fig_freq=self.save_fig_freq, img_folder=self.img_folder, y_th=self.y_th, reg_metric=self.reg_metric, display_metrics=self.display_metrics
                                )
        
        self. kan.results = results
        self.is_fitted_ = True
        # `fit` should always return `self`
        return self

    def predict(self, X):
        """A reference implementation of a predicting function.

        Parameters
        ----------
        X : {array-like, sparse matrix}, shape (n_samples, n_features)
            The training input samples.

        Returns
        -------
        y : ndarray of shape (n_samples, n_outputs)
            The predicted values.
        """
        # Check if fit had been called
        check_is_fitted(self)
        # We need to set reset=False because we don't want to overwrite `n_features_in_`
        # `feature_names_in_` but only check that the shape is consistent.
        X = self._validate_data(X, accept_sparse=True, reset=False)
        
        x = torch.tensor(np.array(X), dtype=torch.float)
        pred = self.kan.forward(x).detach().numpy()
        
        return pred
    

class lmdKANRegressor(RegressorMixin, BaseEstimator):
    """Sci-kit learn wrapper for pykan model.
    
    Hierarchical inheritance chain of classes:
    1. pykan.kan --> 
    2. --> KAN_es: pykan.kan with early stopping, made as in skl.neural_network.MLPRegressor -->
    3. --> KANRegressor: pykan.kan wrapped in (RegressorMixin, BaseEstimator) for compatibility with skl interface.
           Uses params, inspired by skl.neural_network.MLPRegressor params logic.
    """

    # This is a dictionary allowing to define the type of parameters.
    # It used to validate parameter within the `_fit_context` decorator.
    
    _parameter_constraints = {}
    
    _d_solver_translation = {
        'lbfgs': 'LBFGS',
        'adam': 'Adam'
    }

    def __init__(self, 
                 # kan.__init__ args
                 hidden_layer_sizes=None,
                 grid=3,
                 k=3,
                 seed=1,
                 device='cpu',
                 # kan.train_es args
                 tol=1e-3,
                 n_iter_no_change=10,
                 solver='lbfgs',
                 max_iter=100,
                 learning_rate_init=1.0,
                 validation_fraction=0.1,
                 # additional kan.__init__ args
                 mult_arity = 2, noise_scale=0.3, scale_base_mu=0.0, scale_base_sigma=1.0, base_fun='silu', symbolic_enabled=True, affine_trainable=False, grid_eps=0.02, grid_range=[-1, 1], sp_trainable=True, sb_trainable=True, save_act=True, sparse_init=False, auto_save=True, first_init=True, ckpt_path='./model', state_id=0, round=0,
                 lmd_init_vector=None, lmd_init_mu=0.0, lmd_init_sigma=1.0, lmd_trainable=True,
                 # additional kan.train_es args
                 log=1, lamb=0., lamb_l1=1., lamb_entropy=2., lamb_coef=0., lamb_coefdiff=0., update_grid=True, grid_update_num=10, loss_fn=None, start_grid_update_step=-1, stop_grid_update_step=50, batch=-1, metrics=None, singularity_avoiding=False, save_fig=False, in_vars=None, out_vars=None, beta=3, save_fig_freq=1, img_folder='./video',  y_th=1000., reg_metric='edge_forward_spline_n', display_metrics=None
                 ):
        '''
        Init method. Saving all kan.__init__ and kan.fit args.
        
        Args:
        -----
            -- kan.__init__ parameteres --
            hidden_layer_sizes : list
                the ith element represents the number of neurons in the ith hidden layer.
            grid : int
                number of grid intervals. Default: 3.
            k : int
                order of piecewise polynomial. Default: 3.
            mult_arity : int, or list of int lists
                multiplication arity for each multiplication node (the number of numbers to be multiplied)
            noise_scale : float
                initial injected noise to spline.
            base_fun : str
                the residual function b(x). Default: 'silu'
            symbolic_enabled : bool
                compute (True) or skip (False) symbolic computations (for efficiency). By default: True. 
            affine_trainable : bool
                affine parameters are updated or not. Affine parameters include node_scale, node_bias, subnode_scale, subnode_bias
            grid_eps : float
                When grid_eps = 1, the grid is uniform; when grid_eps = 0, the grid is partitioned using percentiles of samples. 0 < grid_eps < 1 interpolates between the two extremes.
            grid_range : list/np.array of shape (2,))
                setting the range of grids. Default: [-1,1]. This argument is not important if fit(update_grid=True) (by default updata_grid=True)
            sp_trainable : bool
                If true, scale_sp is trainable. Default: True.
            sb_trainable : bool
                If true, scale_base is trainable. Default: True.
            device : str
                device
            seed : int
                random seed
            save_act : bool
                indicate whether intermediate activations are saved in forward pass
            sparse_init : bool
                sparse initialization (True) or normal dense initialization. Default: False.
            auto_save : bool
                indicate whether to automatically save a checkpoint once the model is modified
            state_id : int
                the state of the model (used to save checkpoint)
            ckpt_path : str
                the folder to store checkpoints. Default: './model'
            round : int
                the number of times rewind() has been called
            
            -- kan.train_es parameteres --
            tol : float
                Delta of validation fit which doesn`t count as fitness improvement. (Tolerence of training).
            n_iter_no_change : int
                Number of iteration with no fit change to early stopping.
            solver : str
                "lbfgs" or "adam"
            max_iter : int
                training steps
            learning_rate_init: float
                learning rate
            log : int
                logging frequency
            lamb : float
                overall penalty strength
            lamb_l1 : float
                l1 penalty strength
            lamb_entropy : float
                entropy penalty strength
            lamb_coef : float
                coefficient magnitude penalty strength
            lamb_coefdiff : float
                difference of nearby coefficits (smoothness) penalty strength
            update_grid : bool
                If True, update grid regularly before stop_grid_update_step
            grid_update_num : int
                the number of grid updates before stop_grid_update_step
            stop_grid_update_step : int
                no grid updates after this training step
            batch : int
                batch size, if -1 then full.
            small_mag_threshold : float
                threshold to determine large or small numbers (may want to apply larger penalty to smaller numbers)
            small_reg_factor : float
                penalty strength applied to small factors relative to large factos
            save_fig_freq : int
                save figure every (save_fig_freq) step
        '''
        
        self.hidden_layer_sizes, self.grid, self.k, self.seed, self.device = hidden_layer_sizes, grid, k, seed, device
        self.mult_arity, self.noise_scale, self.scale_base_mu, self.scale_base_sigma, self.base_fun, self.symbolic_enabled, self.affine_trainable, self.grid_eps, self.grid_range, self.sp_trainable, self.sb_trainable, self.save_act, self.sparse_init, self.auto_save, self.first_init, self.ckpt_path, self.state_id, self.round = mult_arity, noise_scale, scale_base_mu, scale_base_sigma, base_fun, symbolic_enabled, affine_trainable, grid_eps, grid_range, sp_trainable, sb_trainable, save_act, sparse_init, auto_save, first_init, ckpt_path, state_id, round
        self.lmd_init_vector, self.lmd_init_mu, self.lmd_init_sigma, self.lmd_trainable = lmd_init_vector, lmd_init_mu, lmd_init_sigma, lmd_trainable
        
        self.tol, self.n_iter_no_change, self.solver, self.max_iter, self.learning_rate_init, self.validation_fraction = tol, n_iter_no_change, solver, max_iter, learning_rate_init, validation_fraction
        self.log, self.lamb, self.lamb_l1, self.lamb_entropy, self.lamb_coef, self.lamb_coefdiff, self.update_grid, self.grid_update_num, self.loss_fn, self.start_grid_update_step, self.stop_grid_update_step, self.batch, self.metrics, self.singularity_avoiding, self.save_fig, self.in_vars, self.out_vars, self.beta, self.save_fig_freq, self.img_folder, self.y_th, self.reg_metric, self.display_metrics = log, lamb, lamb_l1, lamb_entropy, lamb_coef, lamb_coefdiff, update_grid, grid_update_num, loss_fn, start_grid_update_step, stop_grid_update_step, batch, metrics, singularity_avoiding, save_fig, in_vars, out_vars, beta, save_fig_freq, img_folder, y_th, reg_metric, display_metrics
        
    @_fit_context(prefer_skip_nested_validation=True)
    def fit(self, X, y, speed=False):
        """A reference implementation of a fitting function.

        Parameters
        ----------
        X : {array-like, sparse matrix}, shape (n_samples, n_features)
            The training input samples.

        y : array-like, shape (n_samples,) or (n_samples, n_outputs)
            The target values (class labels in classification, real numbers in
            regression).

        Returns
        -------
        self : object
            Returns self.
        """
        # `_validate_data` is defined in the `BaseEstimator` class.
        # It allows to:
        # - run different checks on the input data;
        # - define some attributes associated to the input data: `n_features_in_` and
        #   `feature_names_in_`.
        
        # Make sure self.hidden_layer_sizes is a list
        hidden_layer_sizes = self.hidden_layer_sizes
        if not hasattr(hidden_layer_sizes, "__iter__"):
            hidden_layer_sizes = [hidden_layer_sizes]
        hidden_layer_sizes = list(hidden_layer_sizes)
        
        X, y = self._validate_data(X, y, accept_sparse=True)
        n_features = X.shape[1]
        
        # Ensure y is 2D
        if y.ndim == 1:
            y = y.reshape((-1, 1))
            
        if not hasattr(self, "is_fitted_"): self.is_fitted_ = False
        
        if not self.is_fitted_:
            self.n_outputs_ = y.shape[1]
        
            self.width = [n_features] + self.hidden_layer_sizes + [self.n_outputs_]
            self.kan = lmdKAN_es(width=self.width, grid=self.grid, k=self.k, seed=self.seed, device=self.device,
                                 mult_arity=self.mult_arity, noise_scale=self.noise_scale, scale_base_mu=self.scale_base_mu, scale_base_sigma=self.scale_base_sigma, base_fun=self.base_fun, symbolic_enabled=self.symbolic_enabled, affine_trainable=self.affine_trainable, grid_eps=self.grid_eps, grid_range=self.grid_range, sp_trainable=self.sp_trainable, sb_trainable=self.sb_trainable, save_act=self.save_act, sparse_init=self.sparse_init, auto_save=self.auto_save, first_init=self.first_init, ckpt_path=self.ckpt_path, state_id=self.state_id, round=self.round,
                                 lmd_init_vector=self.lmd_init_vector, lmd_init_mu=self.lmd_init_mu, lmd_init_sigma=self.lmd_init_sigma, lmd_trainable=self.lmd_trainable)
        
        if speed: self.kan.speed()
        
        X_train, X_val, y_train, y_val = train_test_split(X, y, 
                                                        test_size=self.validation_fraction,
                                                        random_state=self.seed)
                    
        kan_dataset = {'train_input': torch.tensor(np.array(X_train), dtype=torch.float),
                       'train_label': torch.tensor(np.array(y_train), dtype=torch.float),
                       'test_input': torch.tensor(np.array(X_val), dtype=torch.float),
                       'test_label': torch.tensor(np.array(y_val), dtype=torch.float)}
        
        results = self.kan.fit(kan_dataset, 
                               tol=self.tol, 
                               n_iter_no_change=self.n_iter_no_change,
                               opt=self._d_solver_translation[self.solver], 
                               lr = self.learning_rate_init,
                               steps=self.max_iter,
                               log=self.log, lamb=self.lamb, lamb_l1=self.lamb_l1, lamb_entropy=self.lamb_entropy, lamb_coef=self.lamb_coef, lamb_coefdiff=self.lamb_coefdiff, update_grid=self.update_grid, grid_update_num=self.grid_update_num, loss_fn=self.loss_fn, stop_grid_update_step=self.stop_grid_update_step, batch=self.batch, singularity_avoiding=self.singularity_avoiding, save_fig=self.save_fig, in_vars=self.in_vars, out_vars=self.out_vars, beta=self.beta, save_fig_freq=self.save_fig_freq, img_folder=self.img_folder, y_th=self.y_th, reg_metric=self.reg_metric, display_metrics=self.display_metrics
                                )
        
        self. kan.results = results
        self.is_fitted_ = True
        # `fit` should always return `self`
        return self

    def predict(self, X):
        """A reference implementation of a predicting function.

        Parameters
        ----------
        X : {array-like, sparse matrix}, shape (n_samples, n_features)
            The training input samples.

        Returns
        -------
        y : ndarray of shape (n_samples, n_outputs)
            The predicted values.
        """
        # Check if fit had been called
        check_is_fitted(self)
        # We need to set reset=False because we don't want to overwrite `n_features_in_`
        # `feature_names_in_` but only check that the shape is consistent.
        X = self._validate_data(X, accept_sparse=True, reset=False)
        
        x = torch.tensor(np.array(X), dtype=torch.float)
        pred = self.kan.forward(x).detach().numpy()
        
        return pred

## Loading data

In [3]:
def alg_sklMLP_model(x, y, class_model, model_kwargs, seed = None):
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, 
                                                        train_size=TRAIN_TEST_RATIO,
                                                        random_state=seed)
    
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)

    #print(model_kwargs)
    model = class_model(random_state=seed, **model_kwargs)
    model.fit(x_train, y_train)

    pred_test = model.predict(x_test)
    rmse = mean_squared_error(y_test, pred_test)
    r2 = r2_score(y_test, pred_test)
    mae = mean_absolute_error(y_test, pred_test)
    
    del model

    return [rmse, r2, mae]

def alg_sklKAN_model(x, y, class_model, model_kwargs, seed = None):
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, 
                                                        train_size=TRAIN_TEST_RATIO,
                                                        random_state=seed)
    
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)

    #print(model_kwargs)
    model = class_model(seed=seed, **model_kwargs)
    model.fit(x_train, y_train)

    pred_test = model.predict(x_test)
    rmse = mean_squared_error(y_test, pred_test)
    r2 = r2_score(y_test, pred_test)
    mae = mean_absolute_error(y_test, pred_test)
    
    del model

    return [rmse, r2, mae]


MLP_model_kwargs = {'hidden_layer_sizes': HIDDEN_LAYER_SIZES,
                  'activation': ACTIVATION,
                  'solver': SOLVER,
                  'early_stopping': True,
                  'validation_fraction': VALIDATION_TRAIN_RATIO,
                  'n_iter_no_change': N_ITER_NO_CHANGE,
                  'learning_rate_init': 0.001,
                  'learning_rate': 'adaptive',
                  'max_iter': MAX_ITER,
                  'tol': TOL}

KAN_model_kwargs = {'hidden_layer_sizes': [1,],
                    'grid': 3,
                    'k': 3,
                    'solver': 'lbfgs',
                    'max_iter': 100,
                    'lamb': 1e-2,
                    'lamb_l1': 1,
                    'lamb_entropy': 2,
                    'n_iter_no_change': 30,
                    'tol': 1e-3
                    }

lmdKAN_model_kwargs = {'hidden_layer_sizes': [5,2],
                       'grid': 3,
                       'k': 3,
                       'solver': 'lbfgs',
                       'max_iter': 100,
                       'lamb': 1e-2,
                       'lamb_l1': 1,
                       'lamb_entropy': 2,
                       'n_iter_no_change': 30,
                       'tol': 1e-3
                       }


In [4]:
def multi_exp(l_algos_names,
              l_algos,
              mult_X_Y,
              l_kwargs,
              l_metrics_names,
              num_iter):
    ''' Function, that process algos(X, Y) and returns df of their metrics. 
    '''
    res_list = []

    for alg, (x, y), kwargs, alg_name in zip(l_algos, mult_X_Y, l_kwargs, l_algos_names):
        print(f'--- Processing {alg_name}')

        for i in range(1, num_iter+1):
            print(f'iter: {i}')
            #print(kwargs)
            l_metrics = alg(x, y, seed=i, **kwargs)
            res_list.append([alg_name, i]+l_metrics)
        print('-------')

    return pd.DataFrame(res_list, columns=['alg_name', 'iter']+l_metrics_names)

In [5]:
l_algos_names=['MLP_Cr', 'MLP_Cu', 'MLP_Ni', 'MLP_NO3',
               'KAN_Cr', 'KAN_Cu', 'KAN_Ni', 'KAN_NO3',
               'lmdKAN_Cr', 'lmdKAN_Cu', 'lmdKAN_Ni', 'lmdKAN_NO3']

l_algos=[alg_sklMLP_model, alg_sklMLP_model, alg_sklMLP_model, alg_sklMLP_model,
         alg_sklKAN_model, alg_sklKAN_model, alg_sklKAN_model, alg_sklKAN_model,
         alg_sklKAN_model, alg_sklKAN_model, alg_sklKAN_model, alg_sklKAN_model]

mult_X_Y=[(x, y_Cr), (x, y_Cu), (x, y_Ni), (x, y_NO3), 
          (x, y_Cr), (x, y_Cu), (x, y_Ni), (x, y_NO3),
          (x, y_Cr), (x, y_Cu), (x, y_Ni), (x, y_NO3)]

l_kwargs=[{'class_model': MLPRegressor,'model_kwargs': MLP_model_kwargs},
          {'class_model': MLPRegressor,'model_kwargs': MLP_model_kwargs},
          {'class_model': MLPRegressor,'model_kwargs': MLP_model_kwargs},
          {'class_model': MLPRegressor,'model_kwargs': MLP_model_kwargs},
          
          {'class_model': KANRegressor,'model_kwargs': KAN_model_kwargs},
          {'class_model': KANRegressor,'model_kwargs': KAN_model_kwargs},
          {'class_model': KANRegressor,'model_kwargs': KAN_model_kwargs},
          {'class_model': KANRegressor,'model_kwargs': KAN_model_kwargs},
          
          {'class_model': lmdKANRegressor,'model_kwargs': lmdKAN_model_kwargs},
          {'class_model': lmdKANRegressor,'model_kwargs': lmdKAN_model_kwargs},
          {'class_model': lmdKANRegressor,'model_kwargs': lmdKAN_model_kwargs},
          {'class_model': lmdKANRegressor,'model_kwargs': lmdKAN_model_kwargs},]

l_metrics_names=['rmse', 'r2', 'mae']

num_iter=100


In [6]:
full_df = multi_exp(l_algos_names=l_algos_names,
                    l_algos=l_algos,
                    mult_X_Y=mult_X_Y,
                    l_kwargs=l_kwargs,
                    l_metrics_names=l_metrics_names,
                    num_iter=num_iter)

--- Processing MLP_Cr
iter: 1
iter: 2
iter: 3
iter: 4
iter: 5
iter: 6
iter: 7
iter: 8
iter: 9
iter: 10
iter: 11
iter: 12
iter: 13
iter: 14
iter: 15
iter: 16
iter: 17
iter: 18
iter: 19
iter: 20
iter: 21
iter: 22
iter: 23
iter: 24
iter: 25
iter: 26
iter: 27
iter: 28
iter: 29
iter: 30
iter: 31
iter: 32
iter: 33
iter: 34
iter: 35
iter: 36
iter: 37
iter: 38
iter: 39
iter: 40
iter: 41
iter: 42
iter: 43
iter: 44
iter: 45
iter: 46
iter: 47
iter: 48
iter: 49
iter: 50
iter: 51
iter: 52
iter: 53
iter: 54
iter: 55
iter: 56
iter: 57
iter: 58
iter: 59
iter: 60
iter: 61
iter: 62
iter: 63
iter: 64
iter: 65
iter: 66
iter: 67
iter: 68
iter: 69
iter: 70
iter: 71
iter: 72
iter: 73
iter: 74
iter: 75
iter: 76
iter: 77
iter: 78
iter: 79
iter: 80
iter: 81
iter: 82
iter: 83
iter: 84
iter: 85
iter: 86
iter: 87
iter: 88
iter: 89
iter: 90
iter: 91
iter: 92
iter: 93
iter: 94
iter: 95
iter: 96
iter: 97
iter: 98
iter: 99
iter: 100
-------
--- Processing MLP_Cu
iter: 1
iter: 2
iter: 3
iter: 4
iter: 5
iter: 6
iter: 7


| trn_loss: 3.24e-01 | tst_loss: 3.36e-01 | e_stop: 30/30 | reg: 9.37e+00 | :  94%|▉| 94/100 [00:17<


Early stopping criteria raised
saving model version 0.1
iter: 2
checkpoint directory created: ./model
saving model version 0.0


| trn_loss: 4.07e-01 | tst_loss: 3.66e-01 | e_stop: 30/30 | reg: 7.38e+00 | :  36%|▎| 36/100 [00:05<


Early stopping criteria raised
saving model version 0.1
iter: 3
checkpoint directory created: ./model
saving model version 0.0


...

Early stopping criteria raised
saving model version 0.1
iter: 99
checkpoint directory created: ./model
saving model version 0.0


| trn_loss: 2.78e+00 | tst_loss: 2.94e+00 | e_stop: 30/30 | reg: 4.01e+01 | :  41%|▍| 41/100 [00:21<


Early stopping criteria raised
saving model version 0.1
iter: 100
checkpoint directory created: ./model
saving model version 0.0


| trn_loss: 2.94e+00 | tst_loss: 3.12e+00 | e_stop: 30/30 | reg: 3.98e+01 | :  47%|▍| 47/100 [00:24<

Early stopping criteria raised
saving model version 0.1
-------


In [7]:
full_df

,alg_name,iter,rmse,r2,mae
0,MLP_Cr,1,0.195370,0.912368,0.358914
1,MLP_Cr,2,0.203058,0.918579,0.354085
2,MLP_Cr,3,0.242216,0.903074,0.383426
3,MLP_Cr,4,0.230227,0.911457,0.399531
4,MLP_Cr,5,0.220583,0.907901,0.382985
...,...,...,...,...,...
1195,lmdKAN_NO3,96,10.993112,0.763023,2.836856
1196,lmdKAN_NO3,97,10.271236,0.778028,2.771183
1197,lmdKAN_NO3,98,9.526242,0.783016,2.537381
1198,lmdKAN_NO3,99,9.067707,0.778079,2.515548


In [8]:
full_df.to_excel(f'Accuracy_metrics.xlsx')
#pd.read_excel('full_metrics.xlsx').drop('Unnamed: 0', axis=1)

In [ ]:
aggr_df = full_df.groupby(['alg_name']).agg(["mean", "std"]).drop(['iter'], axis=1)
aggr_df.to_excel(f'aggr_Accuracy_metrics.xlsx')
aggr_df

rmse                  r2                 mae          
                mean       std      mean       std      mean       std
alg_name                                                              
KAN_Cr      0.151094  0.118393  0.939226  0.047426  0.275843  0.081338
KAN_Cu      0.588070  0.160172  0.759886  0.068476  0.569367  0.074292
KAN_NO3     9.127693  1.149694  0.784641  0.031634  2.537344  0.192192
KAN_Ni      2.381816  0.270567  0.036730  0.060717  1.319784  0.094479
MLP_Cr      0.202864  0.029077  0.918372  0.011374  0.359317  0.027571
MLP_Cu      0.633501  0.115562  0.741906  0.048335  0.601832  0.063553
MLP_NO3     9.667228  1.234268  0.772611  0.028910  2.589856  0.193319
MLP_Ni      2.352136  0.252780  0.048711  0.051488  1.319042  0.088436
lmdKAN_Cr   0.161150  0.036202  0.935181  0.014367  0.303765  0.036234
lmdKAN_Cu   0.582567  0.107578  0.762585  0.044511  0.574617  0.054079
lmdKAN_NO3  9.748527  1.202873  0.769885  0.034123  2.610525  0.193037
lmdKAN_Ni   2.485714  0.368029 -0.004646  0.110744  1.351495  0.096215

In [11]:
aggr_df.to_excel(f'aggr_Accuracy_metrics.xlsx')